# 07 - Activity Tracing 및 디버깅

이 노트북에서는 **Agentic Retrieval**의 **Activity 추적** 기능을 활용하여
쿼리 실행 과정을 분석하고 디버깅하는 방법을 학습합니다.

## 📋 학습 내용
1. **Activity 구조 이해** - 검색 활동 로그 분석
2. **Sub-Query 분석** - LLM이 생성한 하위 쿼리 추적
3. **Token 사용량 모니터링** - 비용 최적화
4. **실행 시간 분석** - 성능 병목 식별
5. **디버깅 패턴** - 문제 해결 기법

## ⚠️ 사전 요구사항
- `01-setup_knowledge_base.ipynb` 실행 완료

## 1. 환경 설정

In [ ]:
import os
import json
import time
from datetime import datetime
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import KnowledgeBaseRetrievalClient
from azure.search.documents.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalMediumReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

# 환경 변수 로드
load_dotenv()

# Azure AI Search 설정
search_endpoint = os.environ["SEARCH_ENDPOINT"]
search_api_key = os.environ["SEARCH_ADMIN_KEY"]
search_credential = AzureKeyCredential(search_api_key)

# 리소스 이름
KNOWLEDGE_BASE_NAME = "products-kb"

# 클라이언트 생성
kb_client = KnowledgeBaseRetrievalClient(
    endpoint=search_endpoint,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential
)

print(f"✅ 연결 완료: {search_endpoint}")
print(f"✅ Knowledge Base: {KNOWLEDGE_BASE_NAME}")

## 2. Activity 구조 개요

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                        Activity 구조 개요                                      ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  📊 Activity 타입                                                             ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ queryPlan          │ LLM이 생성한 쿼리 계획 (하위 쿼리 목록)            │  ║
║  │ search             │ 실제 검색 실행 (소스별)                            │  ║
║  │ rerank             │ 시맨틱 재순위 처리                                 │  ║
║  │ answerGeneration   │ 답변 생성 (Answer Synthesis 모드)                  │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  📋 Activity 필드                                                             ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ type               │ Activity 타입                                      │  ║
║  │ startTime          │ 시작 시간 (ISO 8601)                              │  ║
║  │ endTime            │ 종료 시간 (ISO 8601)                              │  ║
║  │ durationMs         │ 소요 시간 (밀리초)                                 │  ║
║  │ subQueries         │ 하위 쿼리 목록 (queryPlan)                         │  ║
║  │ knowledgeSourceName│ 검색 대상 소스 (search)                           │  ║
║  │ documentCount      │ 검색된 문서 수                                     │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  💰 Usage (토큰 사용량)                                                        ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ promptTokens       │ 입력 토큰 수                                       │  ║
║  │ completionTokens   │ 출력 토큰 수                                       │  ║
║  │ totalTokens        │ 전체 토큰 수                                       │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

## 3. Activity 추적 함수 정의

In [ ]:
def execute_with_tracing(query: str, reasoning_effort: str = "medium", top: int = 5) -> dict:
    """
    Activity 추적이 포함된 검색을 실행합니다.
    """
    # Reasoning Effort 매핑
    effort_map = {
        "minimal": KnowledgeRetrievalMinimalReasoningEffort(),
        "low": KnowledgeRetrievalLowReasoningEffort(),
        "medium": KnowledgeRetrievalMediumReasoningEffort()
    }
    
    # 요청 생성
    request = KnowledgeBaseRetrievalRequest(
        query=query,
        top=top,
        retrieval_reasoning_effort=effort_map.get(reasoning_effort, effort_map["medium"]),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA
    )
    
    # 실행 시간 측정
    start_time = time.time()
    response = kb_client.retrieve(request)
    total_time = time.time() - start_time
    
    return {
        "response": response,
        "total_time": total_time,
        "query": query,
        "reasoning_effort": reasoning_effort
    }

print("✅ Activity 추적 함수 정의 완료")

## 4. Activity 상세 분석 함수

In [ ]:
def analyze_activity(result: dict) -> dict:
    """
    Activity 정보를 상세 분석합니다.
    """
    response = result["response"]
    analysis = {
        "query": result["query"],
        "reasoning_effort": result["reasoning_effort"],
        "total_time_ms": result["total_time"] * 1000,
        "activities": [],
        "sub_queries": [],
        "search_stats": [],
        "usage": None
    }
    
    # Activity 분석
    if hasattr(response, 'activity') and response.activity:
        for activity in response.activity:
            act_dict = {
                "type": getattr(activity, 'type', 'unknown'),
                "duration_ms": getattr(activity, 'duration_ms', None),
            }
            
            # Query Plan Activity
            if hasattr(activity, 'sub_queries') and activity.sub_queries:
                act_dict["sub_queries"] = []
                for sq in activity.sub_queries:
                    sq_info = {
                        "query": getattr(sq, 'query', 'N/A'),
                        "target_sources": getattr(sq, 'target_knowledge_sources', [])
                    }
                    act_dict["sub_queries"].append(sq_info)
                    analysis["sub_queries"].append(sq_info)
            
            # Search Activity
            if hasattr(activity, 'knowledge_source_name'):
                act_dict["source"] = activity.knowledge_source_name
                act_dict["document_count"] = getattr(activity, 'document_count', 0)
                analysis["search_stats"].append({
                    "source": activity.knowledge_source_name,
                    "documents": getattr(activity, 'document_count', 0),
                    "duration_ms": getattr(activity, 'duration_ms', None)
                })
            
            analysis["activities"].append(act_dict)
    
    # Usage 분석
    if hasattr(response, 'usage') and response.usage:
        analysis["usage"] = {
            "prompt_tokens": getattr(response.usage, 'prompt_tokens', 0),
            "completion_tokens": getattr(response.usage, 'completion_tokens', 0),
            "total_tokens": getattr(response.usage, 'total_tokens', 0)
        }
    
    # 문서 수
    analysis["document_count"] = len(response.data) if hasattr(response, 'data') and response.data else 0
    
    return analysis

print("✅ Activity 분석 함수 정의 완료")

In [ ]:
def print_activity_report(analysis: dict):
    """
    Activity 분석 보고서를 출력합니다.
    """
    print("\n" + "=" * 70)
    print("📊 ACTIVITY TRACING REPORT")
    print("=" * 70)
    
    # 기본 정보
    print(f"\n📝 쿼리: {analysis['query']}")
    print(f"🧠 Reasoning Effort: {analysis['reasoning_effort']}")
    print(f"⏱️  총 실행 시간: {analysis['total_time_ms']:.2f}ms")
    print(f"📄 반환 문서 수: {analysis['document_count']}")
    
    # Sub-Queries
    if analysis["sub_queries"]:
        print(f"\n{'─' * 70}")
        print("🔍 SUB-QUERIES (LLM이 생성한 하위 쿼리)")
        print(f"{'─' * 70}")
        for i, sq in enumerate(analysis["sub_queries"], 1):
            print(f"\n   [{i}] {sq['query']}")
            if sq.get('target_sources'):
                print(f"       → 대상 소스: {', '.join(sq['target_sources'])}")
    
    # Search Stats
    if analysis["search_stats"]:
        print(f"\n{'─' * 70}")
        print("🔎 SEARCH STATISTICS")
        print(f"{'─' * 70}")
        print(f"\n   {'소스':<30} {'문서 수':<10} {'소요시간':<15}")
        print(f"   {'-' * 55}")
        for stat in analysis["search_stats"]:
            duration = f"{stat['duration_ms']:.0f}ms" if stat['duration_ms'] else "N/A"
            print(f"   {stat['source']:<30} {stat['documents']:<10} {duration:<15}")
    
    # Token Usage
    if analysis["usage"]:
        print(f"\n{'─' * 70}")
        print("💰 TOKEN USAGE")
        print(f"{'─' * 70}")
        usage = analysis["usage"]
        print(f"\n   Prompt Tokens:     {usage['prompt_tokens']:,}")
        print(f"   Completion Tokens: {usage['completion_tokens']:,}")
        print(f"   Total Tokens:      {usage['total_tokens']:,}")
        
        # 예상 비용 계산 (GPT-4 기준 가정)
        # Input: $0.03/1K, Output: $0.06/1K
        est_cost = (usage['prompt_tokens'] * 0.03 + usage['completion_tokens'] * 0.06) / 1000
        print(f"\n   💵 예상 비용 (GPT-4 기준): ${est_cost:.4f}")
    
    # Activity Timeline
    if analysis["activities"]:
        print(f"\n{'─' * 70}")
        print("⏱️  ACTIVITY TIMELINE")
        print(f"{'─' * 70}")
        for i, act in enumerate(analysis["activities"], 1):
            duration = f"{act.get('duration_ms', 0):.0f}ms" if act.get('duration_ms') else "N/A"
            print(f"\n   [{i}] Type: {act['type']}")
            print(f"       Duration: {duration}")
            if 'source' in act:
                print(f"       Source: {act['source']}")
            if 'document_count' in act:
                print(f"       Documents: {act['document_count']}")
    
    print("\n" + "=" * 70)

print("✅ Activity 보고서 함수 정의 완료")

## 5. Activity 추적 실행

In [ ]:
# 테스트 쿼리 실행
TEST_QUERY = "3만원 이하의 겨울용 아기 옷 중에서 블루독베이비나 압소바 브랜드 제품"

print(f"🔍 테스트 쿼리: {TEST_QUERY}")
print("\n실행 중...")

result = execute_with_tracing(
    query=TEST_QUERY,
    reasoning_effort="medium",
    top=5
)

# 분석 실행
analysis = analyze_activity(result)

# 보고서 출력
print_activity_report(analysis)

## 6. Reasoning Effort별 Activity 비교

In [ ]:
# 복잡한 쿼리로 Reasoning Effort 비교
COMPLEX_QUERY = "리뷰가 좋은 유아용 겨울 아우터 중에서 세일 중인 제품을 가격순으로 추천해주세요"

print("#" * 70)
print("🔬 Reasoning Effort별 Activity 비교")
print("#" * 70)
print(f"\n📝 쿼리: {COMPLEX_QUERY}\n")

effort_results = {}

for effort in ["minimal", "low", "medium"]:
    result = execute_with_tracing(
        query=COMPLEX_QUERY,
        reasoning_effort=effort,
        top=5
    )
    effort_results[effort] = analyze_activity(result)
    print(f"✅ {effort.upper()} 완료")

In [ ]:
# 비교 테이블 출력
print("\n" + "=" * 80)
print("📊 REASONING EFFORT 비교 분석")
print("=" * 80)

print(f"\n{'Effort':<12} {'Sub-Queries':<15} {'문서 수':<10} {'시간(ms)':<15} {'토큰':<15}")
print("-" * 70)

for effort, analysis in effort_results.items():
    sub_q_count = len(analysis['sub_queries'])
    doc_count = analysis['document_count']
    time_ms = f"{analysis['total_time_ms']:.0f}"
    tokens = analysis['usage']['total_tokens'] if analysis['usage'] else 'N/A'
    
    print(f"{effort:<12} {sub_q_count:<15} {doc_count:<10} {time_ms:<15} {tokens}")

In [ ]:
# Sub-Query 상세 비교
print("\n" + "=" * 80)
print("🔍 SUB-QUERY 상세 비교")
print("=" * 80)

for effort, analysis in effort_results.items():
    print(f"\n┌─ {effort.upper()} ─────────────────────────────────────────────────────┐")
    if analysis['sub_queries']:
        for i, sq in enumerate(analysis['sub_queries'], 1):
            print(f"│  [{i}] {sq['query'][:55]}..." if len(sq['query']) > 55 else f"│  [{i}] {sq['query']}")
    else:
        print("│  (Sub-query 생성 없음 - 단일 쿼리 실행)")
    print(f"└{'─' * 65}┘")

## 7. 토큰 사용량 모니터링

In [ ]:
class TokenMonitor:
    """
    토큰 사용량을 모니터링하는 클래스입니다.
    """
    
    def __init__(self):
        self.history = []
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        
    def record(self, query: str, analysis: dict):
        """사용량을 기록합니다."""
        if analysis['usage']:
            record = {
                "timestamp": datetime.now().isoformat(),
                "query": query[:50] + "..." if len(query) > 50 else query,
                "reasoning_effort": analysis['reasoning_effort'],
                "prompt_tokens": analysis['usage']['prompt_tokens'],
                "completion_tokens": analysis['usage']['completion_tokens'],
                "total_tokens": analysis['usage']['total_tokens']
            }
            self.history.append(record)
            self.total_prompt_tokens += analysis['usage']['prompt_tokens']
            self.total_completion_tokens += analysis['usage']['completion_tokens']
    
    def get_summary(self) -> dict:
        """사용량 요약을 반환합니다."""
        return {
            "total_requests": len(self.history),
            "total_prompt_tokens": self.total_prompt_tokens,
            "total_completion_tokens": self.total_completion_tokens,
            "total_tokens": self.total_prompt_tokens + self.total_completion_tokens,
            "avg_tokens_per_request": (self.total_prompt_tokens + self.total_completion_tokens) / len(self.history) if self.history else 0
        }
    
    def print_report(self):
        """사용량 보고서를 출력합니다."""
        summary = self.get_summary()
        
        print("\n" + "=" * 70)
        print("💰 TOKEN USAGE MONITORING REPORT")
        print("=" * 70)
        
        print(f"\n📈 전체 요약")
        print(f"   총 요청 수: {summary['total_requests']}")
        print(f"   총 토큰 수: {summary['total_tokens']:,}")
        print(f"   평균 토큰/요청: {summary['avg_tokens_per_request']:.0f}")
        
        # 비용 계산 (GPT-4 기준)
        est_cost = (summary['total_prompt_tokens'] * 0.03 + summary['total_completion_tokens'] * 0.06) / 1000
        print(f"\n💵 예상 비용 (GPT-4 기준): ${est_cost:.4f}")
        
        print(f"\n📋 요청 히스토리")
        print(f"{'─' * 70}")
        print(f"{'Effort':<10} {'Tokens':<10} {'Query':<45}")
        print(f"{'─' * 70}")
        for record in self.history:
            print(f"{record['reasoning_effort']:<10} {record['total_tokens']:<10} {record['query']}")

# 모니터 인스턴스 생성
token_monitor = TokenMonitor()

print("✅ Token Monitor 클래스 정의 완료")

In [ ]:
# 여러 쿼리 실행 및 모니터링
test_queries = [
    ("아기 옷 추천", "minimal"),
    ("겨울용 따뜻한 유아복", "low"),
    ("블루독베이비 3만원 이하 제품", "low"),
    ("리뷰 좋은 압소바 겨울 아우터 세일 제품", "medium"),
    ("신생아용 선물 세트 추천해주세요", "medium")
]

print("🔍 여러 쿼리 실행 중...\n")

for query, effort in test_queries:
    result = execute_with_tracing(query=query, reasoning_effort=effort)
    analysis = analyze_activity(result)
    token_monitor.record(query, analysis)
    print(f"  ✅ [{effort}] {query[:40]}...")

In [ ]:
# 토큰 사용량 보고서 출력
token_monitor.print_report()

## 8. 실행 시간 분석

In [ ]:
def analyze_performance(queries: list, iterations: int = 3) -> dict:
    """
    여러 쿼리의 성능을 분석합니다.
    """
    results = []
    
    for query, effort in queries:
        times = []
        for _ in range(iterations):
            result = execute_with_tracing(query=query, reasoning_effort=effort)
            times.append(result['total_time'] * 1000)
        
        results.append({
            "query": query[:40] + "..." if len(query) > 40 else query,
            "effort": effort,
            "min_ms": min(times),
            "max_ms": max(times),
            "avg_ms": sum(times) / len(times)
        })
    
    return results

print("✅ 성능 분석 함수 정의 완료")

In [ ]:
# 성능 분석 실행
print("⏱️  성능 분석 실행 중... (각 쿼리 3회 반복)\n")

perf_queries = [
    ("아기 옷", "minimal"),
    ("아기 옷", "low"),
    ("아기 옷", "medium"),
]

perf_results = analyze_performance(perf_queries, iterations=3)

In [ ]:
# 성능 분석 결과 출력
print("\n" + "=" * 70)
print("⏱️  PERFORMANCE ANALYSIS REPORT")
print("=" * 70)

print(f"\n{'Query':<30} {'Effort':<10} {'Min(ms)':<12} {'Max(ms)':<12} {'Avg(ms)':<12}")
print("-" * 70)

for r in perf_results:
    print(f"{r['query']:<30} {r['effort']:<10} {r['min_ms']:.0f}{'':<8} {r['max_ms']:.0f}{'':<8} {r['avg_ms']:.0f}")

# 성능 차이 분석
if len(perf_results) >= 3:
    minimal_avg = perf_results[0]['avg_ms']
    low_avg = perf_results[1]['avg_ms']
    medium_avg = perf_results[2]['avg_ms']
    
    print(f"\n📊 Reasoning Effort 증가에 따른 지연 시간:")
    print(f"   minimal → low:    +{((low_avg - minimal_avg) / minimal_avg * 100):.1f}%")
    print(f"   low → medium:     +{((medium_avg - low_avg) / low_avg * 100):.1f}%")
    print(f"   minimal → medium: +{((medium_avg - minimal_avg) / minimal_avg * 100):.1f}%")

## 9. 디버깅 패턴

In [ ]:
def debug_query(query: str, reasoning_effort: str = "medium") -> None:
    """
    쿼리 실행을 단계별로 디버깅합니다.
    """
    print("\n" + "#" * 70)
    print("🐛 DEBUG MODE")
    print("#" * 70)
    
    print(f"\n📝 Input Query: {query}")
    print(f"🧠 Reasoning Effort: {reasoning_effort}")
    
    # Step 1: 쿼리 실행
    print(f"\n{'─' * 50}")
    print("📍 STEP 1: 쿼리 실행")
    print(f"{'─' * 50}")
    
    start = time.time()
    result = execute_with_tracing(query=query, reasoning_effort=reasoning_effort)
    execution_time = (time.time() - start) * 1000
    
    print(f"   ✅ 실행 완료: {execution_time:.0f}ms")
    
    response = result['response']
    
    # Step 2: Activity 확인
    print(f"\n{'─' * 50}")
    print("📍 STEP 2: Activity 확인")
    print(f"{'─' * 50}")
    
    if hasattr(response, 'activity') and response.activity:
        print(f"   Activity 수: {len(response.activity)}")
        for i, act in enumerate(response.activity, 1):
            act_type = getattr(act, 'type', 'unknown')
            print(f"   [{i}] Type: {act_type}")
    else:
        print("   ⚠️ Activity 정보 없음")
    
    # Step 3: Sub-Query 확인
    print(f"\n{'─' * 50}")
    print("📍 STEP 3: Sub-Query 확인")
    print(f"{'─' * 50}")
    
    sub_queries_found = False
    if hasattr(response, 'activity') and response.activity:
        for act in response.activity:
            if hasattr(act, 'sub_queries') and act.sub_queries:
                sub_queries_found = True
                print(f"   Sub-Query 수: {len(act.sub_queries)}")
                for j, sq in enumerate(act.sub_queries, 1):
                    print(f"   [{j}] {getattr(sq, 'query', 'N/A')}")
    
    if not sub_queries_found:
        print("   ℹ️ Sub-Query 없음 (단일 쿼리 실행)")
    
    # Step 4: 결과 확인
    print(f"\n{'─' * 50}")
    print("📍 STEP 4: 결과 확인")
    print(f"{'─' * 50}")
    
    if hasattr(response, 'data') and response.data:
        print(f"   반환 문서 수: {len(response.data)}")
        print(f"\n   상위 3개 문서:")
        for i, doc in enumerate(response.data[:3], 1):
            title = doc.get('title', 'N/A')[:45]
            score = doc.get('@search.reranker_score', doc.get('@search.score', 'N/A'))
            print(f"   [{i}] {title}")
            print(f"       Score: {score}")
    else:
        print("   ⚠️ 검색 결과 없음")
    
    # Step 5: 토큰 사용량
    print(f"\n{'─' * 50}")
    print("📍 STEP 5: 토큰 사용량")
    print(f"{'─' * 50}")
    
    if hasattr(response, 'usage') and response.usage:
        print(f"   Prompt Tokens: {getattr(response.usage, 'prompt_tokens', 'N/A')}")
        print(f"   Completion Tokens: {getattr(response.usage, 'completion_tokens', 'N/A')}")
        print(f"   Total Tokens: {getattr(response.usage, 'total_tokens', 'N/A')}")
    else:
        print("   ℹ️ 토큰 사용량 정보 없음 (minimal effort)")
    
    print(f"\n{'─' * 50}")
    print("✅ DEBUG 완료")
    print(f"{'─' * 50}")

print("✅ 디버깅 함수 정의 완료")

In [ ]:
# 디버깅 실행 예제
debug_query(
    query="블루독베이비 겨울 아우터 3만원 이하",
    reasoning_effort="medium"
)

## 10. 문제 해결 가이드

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║                        🔧 문제 해결 가이드                                     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  ❌ 문제: 검색 결과가 없거나 적음                                              ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ 1. reranker_threshold 낮추기 (0.0 ~ 1.0)                               │  ║
║  │ 2. reasoning_effort 높이기 (low → medium)                              │  ║
║  │ 3. 쿼리 단순화 (복잡한 조건 분리)                                        │  ║
║  │ 4. top 파라미터 증가                                                    │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  ❌ 문제: 응답 시간이 너무 느림                                                ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ 1. reasoning_effort 낮추기 (medium → low → minimal)                    │  ║
║  │ 2. max_sub_queries 줄이기                                              │  ║
║  │ 3. max_runtime_seconds 설정                                            │  ║
║  │ 4. top 파라미터 감소                                                    │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  ❌ 문제: 토큰 사용량이 너무 많음                                              ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ 1. reasoning_effort 낮추기                                             │  ║
║  │ 2. max_output_size 제한                                                │  ║
║  │ 3. 단순 쿼리는 minimal effort 사용                                      │  ║
║  │ 4. 캐싱 전략 구현                                                       │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  ❌ 문제: 검색 결과 품질이 낮음                                                ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ 1. reranker_threshold 높이기 (2.0 ~ 3.0)                               │  ║
║  │ 2. reasoning_effort 높이기                                             │  ║
║  │ 3. Knowledge Source 설정 검토                                          │  ║
║  │ 4. 인덱스 필드 및 설정 검토                                             │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
║  📋 디버깅 체크리스트                                                          ║
║  ┌────────────────────────────────────────────────────────────────────────┐  ║
║  │ □ Activity 로그 확인 (쿼리 계획, 실행 시간)                              │  ║
║  │ □ Sub-Query 분석 (LLM이 쿼리를 어떻게 분해했는지)                        │  ║
║  │ □ 토큰 사용량 모니터링                                                  │  ║
║  │ □ 검색 통계 확인 (소스별 문서 수, 시간)                                  │  ║
║  │ □ Reranker 점수 분포 확인                                              │  ║
║  └────────────────────────────────────────────────────────────────────────┘  ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

## 11. 정리

이 노트북에서 학습한 내용:

1. **Activity 구조**: 쿼리 실행 로그의 구조와 의미
2. **Sub-Query 분석**: LLM이 생성한 하위 쿼리 추적
3. **Token 모니터링**: 사용량 추적 및 비용 최적화
4. **성능 분석**: 실행 시간 측정 및 병목 식별
5. **디버깅 패턴**: 단계별 문제 해결 접근법

In [ ]:
print("\n" + "=" * 70)
print("✅ Activity Tracing 및 디버깅 실습 완료!")
print("=" * 70)
print("""
다음 노트북:
- 08-answer_synthesis.ipynb: Answer Instructions 커스터마이징
""")